# Milestone 3 Pushshift Reddit Preprocessing & First Model Building and Evaluation

This notebook performs preprocessing and preliminary model building and evaluation on the Pushshift Reddit dataset.

In [2]:
# Dependencies 

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import LongType
import requests

Matplotlib created a temporary cache directory at /scratch/ekim18/job_48810757/matplotlib-p77p22n0 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [3]:
# SparkSession Configuration

# 16 cores, 128GB total memory — local[*] mode
# In local mode there are no separate executor processes — all task execution
# runs as threads within a single JVM. Executor config parameters have no effect
# and are omitted. Driver memory is set to 120GB to give the JVM nearly the
# full node allocation.
spark = SparkSession.builder \
    .appName("PushshiftRedditPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "120g") \
    .config("spark.driver.maxResultSize", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.local.dir", "/expanse/lustre/projects/uci157/ekim18/spark-tmp") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark version: 3.5.0
Spark UI: http://exp-1-44.expanse.sdsc.edu:4040


## Data Loading and Preprocessing Pipeline

Raw data is loaded from `data/raw/` using the pre/post 2015 cutoff fix to handle the `created_utc` schema inconsistency across Parquet files (files before RS_2015-01 store `created_utc` as BINARY/STRING while later files store it as INT64). The two file groups are read separately with an explicit `.cast('long')` on `created_utc` then unioned into a single DataFrame.

`row_count` is hardcoded from the EDA notebook to avoid re-triggering a full dataset scan on every kernel restart.

In [3]:
# Data Load 

DATA_DIR = "../data/raw/"
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))

COLS = ["author", "created_utc", "id", "num_comments", "score",
        "selftext", "subreddit", "subreddit_id", "title"]

pre_cutoff = [f for f in files if os.path.basename(f) >= "RS_2015-01"]
post_cutoff = [f for f in files if os.path.basename(f) < "RS_2015-01"]

# Files where created_utc is STRING - need to cast
df_pre = spark.read.parquet(*pre_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

df_post = spark.read.parquet(*post_cutoff) \
    .select(*[F.col(c) for c in COLS if c != 'created_utc'],
            F.col('created_utc').cast('long').alias('created_utc'))

MIN_TS = 1119398400  # June 2005
MAX_TS = 1700000000  # Nov 2023

df = df_pre.union(df_post) \
    .filter(F.col('created_utc').between(MIN_TS, MAX_TS))

print(f"Files loaded: {len(files)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Files loaded: 218
Partitions: 683


In [4]:
# SparkUI Screenshot

# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
sparkUI_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
sparkUI_df['maxMemory_GB'] = (sparkUI_df['maxMemory'] / (1024**3)).round(2)
print(sparkUI_df)

# Spark context master check
print(spark.sparkContext.master)

       id  totalCores    maxMemory  activeTasks  isActive  maxMemory_GB
0  driver          16  77120667648            0      True         71.82
local[*]


In [5]:
# Hardcode row_count to dataset size for count verifications

row_count = 549662955

## Filtering and Cleaning

The following filters are applied to remove invalid records before any feature engineering or label generation. Decisions on what to filter and what to retain as features are documented in the EDA notebook (`notebooks/Milestone2_Pushshift.ipynb`).

| Filter | Rows Removed | Reason |
|---|---|---|
| Null `subreddit` / `subreddit_id` | ~306,479 | Cannot contribute to per-subreddit features |
| Null `score` | 21 | Required for label generation |
| Negative `num_comments` | 1,090 | Pushshift artifact, value unverifiable |

Zero-length titles, deleted/empty authors, and bot authors are **not** filtered. EDA showed these groups have distinct and meaningful engagement patterns that are captured as binary features (`has_title`, `is_anonymous_author`, `is_known_bot`) instead.

EDA identified exactly 1 duplicate post ID across 549M rows. Deduplication was omitted from the preprocessing pipeline as `dropDuplicates` requires a full shuffle across the entire dataset which is an expensive operation that is not justified for removing a single row with no meaningful impact on model training.

In [6]:
# ── FILTERING & CLEANING ─────────────────────────────────────────────────────

df_clean = df \
    .filter(F.col("subreddit").isNotNull()) \
    .filter(F.col("subreddit_id").isNotNull()) \
    .filter(F.col("score").isNotNull()) \
    .filter(F.col("num_comments") >= 0)

print(f"Original row count:      {row_count:,}")
clean_count = df_clean.count()
print(f"Row count after filters: {clean_count:,}")
print(f"Rows removed:            {row_count - clean_count:,}")

Original row count:      549,662,955
Row count after filters: 549,355,365
Rows removed:            307,590


## Bot Author Handling

Rather than attempting frequency-based bot detection (which would require a full groupBy on author + date across 549M rows to compute per-author daily post rates), we take a simpler and more interpretable approach: flagging known high-volume bot accounts identified in EDA as a binary feature `is_known_bot`.

This preserves all bot-authored posts in the dataset — bot posts have real and consistent engagement patterns (clustering heavily in low-engagement) that the model can learn from. Filtering them out would discard genuine signal. The `is_known_bot` flag gives the model explicit information about author type without requiring expensive per-author temporal aggregations.

The known bot list is seeded from the top authors output in EDA. Frequency-based detection can be revisited in as feature engineering improvement.

In [7]:
# ── BOT AUTHOR FLAGGING ───────────────────────────────────────────────────────

KNOWN_BOTS = {
    "AutoModerator", "AutoNewsAdmin", "AutoNewspaperAdmin",
    "politicbot", "RPBot", "ImagesOfNetwork", "-en-",
    "KellyfromLeedsUK"
}

df_clean = df_clean \
    .withColumn("is_known_bot",
        F.when(F.col("author").isin(list(KNOWN_BOTS)), 1)
         .otherwise(0)) \
    .withColumn("is_anonymous_author",
        F.when(
            (F.col("author") == "[deleted]") | (F.col("author") == ""), 1)
         .otherwise(0)) \
    .withColumn("has_title",
        F.when(F.length(F.col("title")) == 0, 0)
         .otherwise(1))

print("Author type features added.")
df_clean.select("is_known_bot", "is_anonymous_author", "has_title").show(5)

Author type features added.
+------------+-------------------+---------+
|is_known_bot|is_anonymous_author|has_title|
+------------+-------------------+---------+
|           0|                  1|        1|
|           0|                  1|        1|
|           0|                  0|        1|
|           0|                  1|        1|
|           0|                  0|        1|
+------------+-------------------+---------+
only showing top 5 rows



#### Distribution of subreddit post counts

First, we need to understand the shape of the subreddit post count distributions. 

In [8]:
subreddit_post_counts = df_clean \
    .groupBy("subreddit") \
    .count() \
    .withColumnRenamed("count", "post_count")

subreddit_post_counts.select("post_count").describe().show()

+-------+------------------+
|summary|        post_count|
+-------+------------------+
|  count|           2291802|
|   mean|239.70454908408317|
| stddev|16399.513624659332|
|    min|                 1|
|    max|          14521450|
+-------+------------------+



And the distribution at the low end specifically.

In [9]:
subreddit_post_counts.groupBy(
    F.when(F.col("post_count") < 10, "<10")
     .when(F.col("post_count") < 50, "10-49")
     .when(F.col("post_count") < 100, "50-99")
     .when(F.col("post_count") < 500, "100-499")
     .when(F.col("post_count") < 1000, "500-999")
     .otherwise("1000+")
     .alias("post_count_bucket")
) \
.count() \
.orderBy("post_count_bucket") \
.show()

+-----------------+-------+
|post_count_bucket|  count|
+-----------------+-------+
|            10-49| 291866|
|          100-499|  62694|
|            1000+|  26624|
|            50-99|  51649|
|          500-999|  13421|
|              <10|1845548|
+-----------------+-------+



### What percentage of total posts would be excluded at various thresholds?

In [10]:
for threshold in [10, 50, 100, 500, 1000]:
    kept = subreddit_post_counts.filter(F.col("post_count") >= threshold)
    n_subreddits = kept.count()
    n_posts = kept.agg(F.sum("post_count")).collect()[0][0]
    print(f"Threshold {threshold:5d}: {n_subreddits:8,} subreddits kept, {n_posts:12,} posts kept ({n_posts/549355364*100:.1f}%)")

Threshold    10:  446,254 subreddits kept,  545,144,030 posts kept (99.2%)
Threshold    50:  154,388 subreddits kept,  539,079,288 posts kept (98.1%)
Threshold   100:  102,739 subreddits kept,  535,480,818 posts kept (97.5%)
Threshold   500:   40,045 subreddits kept,  521,649,828 posts kept (95.0%)
Threshold  1000:   26,624 subreddits kept,  512,178,009 posts kept (93.2%)


## Subreddit Minimum Post Count Filter

Per-subreddit percentile thresholds are only statistically meaningful when a subreddit
has enough posts for `percentile_approx` to produce a reliable estimate. Subreddits
with very few posts produce degenerate thresholds — for example, a subreddit with 5
posts all scoring 1 has a median of 1, meaning any post with score ≥ 1 clears the
"high score" threshold, which is essentially every post on Reddit.

Analysis of the subreddit post count distribution reveals that 1,845,548 subreddits
(80.5% of all 2,291,802 subreddits) have fewer than 10 posts, and the distribution is
extremely long-tailed (mean: 239, stddev: 16,399). The vast majority of actual Reddit
activity is concentrated in a small number of active communities.

Post retention at various minimum thresholds:

| Min Posts | Subreddits Kept | Posts Kept |
|---|---|---|
| 10 | 446,254 | 99.2% |
| 50 | 154,388 | 98.1% |
| 100 | 102,739 | 97.5% |
| 500 | 40,045 | 95.0% |
| 1000 | 26,624 | 93.2% |

We select a minimum of **100 posts per subreddit** as the threshold. At this cutoff,
`percentile_approx` has sufficient data to produce stable estimates, the jump from 50
to 100 costs only 0.6% of posts while eliminating 51,649 additional unreliable
subreddits, and 97.5% post retention ensures the dataset remains representative.
Subreddits below this threshold are excluded from label generation entirely.

In [11]:
# ── SUBREDDIT MINIMUM POST COUNT FILTER ──────────────────────────────────────

MIN_SUBREDDIT_POSTS = 100

# Compute per-subreddit post counts and filter to active subreddits
active_subreddits = df_clean \
    .groupBy("subreddit") \
    .count() \
    .filter(F.col("count") >= MIN_SUBREDDIT_POSTS) \
    .select("subreddit")

df_active = df_clean.join(active_subreddits, on="subreddit", how="inner")

active_count = df_active.count()
print(f"Rows after subreddit filter: {active_count:,}")
print(f"Rows removed:                {clean_count - active_count:,}")
print(f"Posts retained:              {active_count / clean_count * 100:.1f}%")

Rows after subreddit filter: 535,480,818
Rows removed:                13,874,547
Posts retained:              97.5%


## Label Generation — Threshold Sensitivity Analysis

Before committing to a percentile threshold for engagement archetype assignment, we
evaluated how the class distribution shifts across a range of thresholds. The goal is
to find a threshold that produces a label distribution that reflects the reality of
Reddit engagement — truly viral posts should be rare, and low-engagement posts should
be the majority.

We ruled out standard deviation-based thresholds despite their statistical appeal
because Reddit score distributions are heavily right-skewed (global stddev: 707,
mean: 44.8), not normally distributed. Standard deviation thresholds assume roughly
normal distributions and behave unpredictably on skewed data. Percentile-based
thresholds are distribution-agnostic and more robust for this use case.

Sensitivity analysis was run exploratorily across percentiles 0.5 through 0.6 before
hitting compute constraints. Results:

| Percentile | viral | crowd-pleaser | debate-starter | low-engagement |
|---|---|---|---|---|
| 0.50 | 296,939,301 (55%) | 81,648,029 (15%) | 78,517,491 (15%) | 78,375,997 (15%) |
| 0.60 | 237,653,516 (44%) | 90,612,364 (17%) | 74,714,210 (14%) | 132,500,728 (25%) |

At both thresholds the viral class is the dominant class — the opposite of expected.
The trend shows viral dropping ~11% and low-engagement growing ~10% per 0.1 increment.
We select **0.75** as the final threshold, extrapolating that it brings the distribution
to a more intuitive shape where low-engagement is the plurality class. The 0.75
threshold is also conceptually clean — a post must beat 3 out of 4 posts in its
subreddit on both axes simultaneously to qualify as viral.

In [12]:
# ── LABEL GENERATION — 75TH PERCENTILE THRESHOLD ─────────────────────────────

SCORE_PERCENTILE = 0.75
COMMENTS_PERCENTILE = 0.75

subreddit_thresholds = df_active \
    .groupBy("subreddit") \
    .agg(
        F.expr(f"percentile_approx(score, {SCORE_PERCENTILE})").alias("score_thresh"),
        F.expr(f"percentile_approx(num_comments, {COMMENTS_PERCENTILE})").alias("comments_thresh")
    )

df_labeled = df_active.join(subreddit_thresholds, on="subreddit", how="left") \
    .withColumn("label_4class",
        F.when(
            (F.col("score") >= F.col("score_thresh")) &
            (F.col("num_comments") >= F.col("comments_thresh")), "viral"
        ).when(
            (F.col("score") >= F.col("score_thresh")) &
            (F.col("num_comments") < F.col("comments_thresh")), "crowd-pleaser"
        ).when(
            (F.col("score") < F.col("score_thresh")) &
            (F.col("num_comments") >= F.col("comments_thresh")), "debate-starter"
        ).otherwise("low-engagement")) \
    .withColumn("label_binary",
        F.when(
            (F.col("score") >= F.col("score_thresh")) |
            (F.col("num_comments") >= F.col("comments_thresh")),
            "high-engagement"
        ).otherwise("low-engagement"))

print("4-class label distribution:")
df_labeled.groupBy("label_4class") \
    .count() \
    .withColumn("pct", F.round(F.col("count") / F.lit(active_count) * 100, 1)) \
    .orderBy(F.col("count").desc()) \
    .show()

print("Binary label distribution:")
df_labeled.groupBy("label_binary") \
    .count() \
    .withColumn("pct", F.round(F.col("count") / F.lit(active_count) * 100, 1)) \
    .orderBy(F.col("count").desc()) \
    .show()

4-class label distribution:
+--------------+---------+----+
|  label_4class|    count| pct|
+--------------+---------+----+
|low-engagement|236954268|44.3|
|         viral|160168118|29.9|
| crowd-pleaser| 74916114|14.0|
|debate-starter| 63442318|11.8|
+--------------+---------+----+

Binary label distribution:
+---------------+---------+----+
|   label_binary|    count| pct|
+---------------+---------+----+
|high-engagement|298526550|55.7|
| low-engagement|236954268|44.3|
+---------------+---------+----+



### Persist Labeled DataFrame 

We persist df_labeled to disk to break expensive lineage and have every subsequent action read from persisted Parquet file to avoid future re-shuffles and avoid disk spills.

In [13]:
# ── PERSIST df_labeled TO data/processed/ ─────────────────────────────────────────────

PROCESSED_DIR = "../data/processed/"

# Drop threshold columns before persisting — not needed downstream
df_labeled = df_labeled.drop("score_thresh", "comments_thresh")

# Write to Parquet
df_labeled.write \
    .mode("overwrite") \
    .parquet(PROCESSED_DIR)

print("Write complete. Reloading from disk...")

# Reload from disk — this breaks the lineage entirely
df_labeled = spark.read.parquet(PROCESSED_DIR)

row_count_labeled = df_labeled.count()
print(f"Rows in persisted df_labeled: {row_count_labeled:,}")

Write complete. Reloading from disk...
Rows in persisted df_labeled: 535,480,818


## Feature Engineering

All features are derived from pre-publication information only to prevent data leakage and no features use `score`, `num_comments`, or the derived label columns.

### Title and Post Type Features

Simple structural features extracted from the `title` and `selftext` columns using PySpark string functions. These require no aggregations or UDFs and are computed directly as column transformations.

- `title_len`: character count of the post title
- `title_word_count`: number of whitespace-delimited words in the title
- `has_question`: binary flag indicating the presence of `?` in the title
- `has_exclamation`: binary flag indicating the presence of `!` in the title
- `is_text_post`: binary flag, 1 if the post has a non-empty selftext body
- `selftext_len`: character length of selftext (0 for link posts)

In [14]:
# Reload df_labeled from data/processed/ 
# Note: df_labeled is loaded in persist cell just for count validation
# This cell begins checkpoint for feature engineering

PROCESSED_DIR = "../data/processed/"
df_labeled = spark.read.parquet(PROCESSED_DIR)

In [15]:
# ── TITLE AND POST TYPE FEATURES ─────────────────────────────────────────────

df_features = df_labeled \
    .withColumn("title_len",
        F.length(F.col("title"))) \
    .withColumn("title_word_count",
        F.when(F.length(F.col("title")) == 0, 0)
         .otherwise(F.size(F.split(F.trim(F.col("title")), r"\s+")))) \
    .withColumn("has_question",
        F.when(F.col("title").contains("?"), 1).otherwise(0)) \
    .withColumn("has_exclamation",
        F.when(F.col("title").contains("!"), 1).otherwise(0)) \
    .withColumn("is_text_post",
        F.when(F.col("selftext") != "", 1).otherwise(0)) \
    .withColumn("selftext_len",
        F.when(F.col("selftext") != "", F.length(F.col("selftext"))).otherwise(0))

df_features.select(
    "title", "title_len", "title_word_count",
    "has_question", "has_exclamation",
    "is_text_post", "selftext_len"
).show(5, truncate=40)

+-----------------------------+---------+----------------+------------+---------------+------------+------------+
|                        title|title_len|title_word_count|has_question|has_exclamation|is_text_post|selftext_len|
+-----------------------------+---------+----------------+------------+---------------+------------+------------+
|          flag wallpaper dump|       19|               3|           0|              0|           0|           0|
|shaktis shoes by jaime ibarra|       29|               5|           0|              0|           0|           0|
|  real love stories by thezgi|       27|               5|           0|              0|           0|           0|
|        everything is nothing|       21|               3|           0|              0|           0|           0|
|             labrys wallpaper|       16|               2|           0|              0|           0|           0|
+-----------------------------+---------+----------------+------------+---------------+-

### Temporal Features

Hour of day and day of week extracted from the `created_utc` Unix timestamp using `F.from_unixtime`. These capture posting time patterns. Our EDA showed a clear weekday peak (Mon–Thu) and lower weekend activity, suggesting temporal features carry meaningful signal for engagement prediction.

- `hour_of_day`: hour of post creation (0–23)
- `day_of_week`: day of week of post creation (1=Sunday, 7=Saturday in Spark)

In [16]:
# ── TEMPORAL FEATURES ────────────────────────────────────────────────────────

df_features = df_features \
    .withColumn("hour_of_day",
        F.hour(F.from_unixtime(F.col("created_utc")))) \
    .withColumn("day_of_week",
        F.dayofweek(F.from_unixtime(F.col("created_utc"))))

df_features.select(
    "created_utc", "hour_of_day", "day_of_week"
).show(5)

+-----------+-----------+-----------+
|created_utc|hour_of_day|day_of_week|
+-----------+-----------+-----------+
| 1420619098|          8|          4|
| 1420619661|          8|          4|
| 1420631012|         11|          4|
| 1420631183|         11|          4|
| 1420631826|         11|          4|
+-----------+-----------+-----------+
only showing top 5 rows



### Subreddit Context Features

Per-subreddit aggregations capturing the behavioral norms of each community. These features give the model context about the subreddit a post was submitted to without directly one-hot encoding the high-cardinality `subreddit` column (~100K active subreddits). Computed via `groupBy("subreddit").agg()` and joined back to the main DataFrame.

- `subreddit_post_count`: total number of posts in the subreddit
- `subreddit_median_score`: median score across all posts in the subreddit
- `subreddit_median_comments`: median comment count across all posts in the subreddit

In [17]:
# ── SUBREDDIT CONTEXT FEATURES ───────────────────────────────────────────────

subreddit_stats = df_features \
    .groupBy("subreddit") \
    .agg(
        F.count("*").alias("subreddit_post_count"),
        F.expr("percentile_approx(score, 0.5)").alias("subreddit_median_score"),
        F.expr("percentile_approx(num_comments, 0.5)").alias("subreddit_median_comments")
    )

df_features = df_features.join(subreddit_stats, on="subreddit", how="left")

df_features.select(
    "subreddit", "subreddit_post_count",
    "subreddit_median_score", "subreddit_median_comments"
).show(5)

+---------+--------------------+----------------------+-------------------------+
|subreddit|subreddit_post_count|subreddit_median_score|subreddit_median_comments|
+---------+--------------------+----------------------+-------------------------+
|    0x10c|                1614|                    10|                        9|
|    0x10c|                1614|                    10|                        9|
|    0x10c|                1614|                    10|                        9|
|    0x10c|                1614|                    10|                        9|
|    0x10c|                1614|                    10|                        9|
+---------+--------------------+----------------------+-------------------------+
only showing top 5 rows



### Author History Features

Per-author aggregations capturing posting behavior and historical performance. Since `author` has ~35M unique values, direct encoding is not feasible. Instead, each author is represented through aggregated numerical features that capture their posting history. Anonymous and deleted authors will have these features computed on their pooled history as a group as their `is_anonymous_author` flag already captures their identity status.

- `author_post_count`: total number of posts by this author in the dataset
- `author_mean_score`: mean score across all of the author's posts

In [18]:
# ── AUTHOR HISTORY FEATURES ──────────────────────────────────────────────────

author_stats = df_features \
    .groupBy("author") \
    .agg(
        F.count("*").alias("author_post_count"),
        F.mean("score").alias("author_mean_score")
    )

df_features = df_features.join(author_stats, on="author", how="left")

df_features.select(
    "author", "author_post_count", "author_mean_score"
).show(5)

+---------------+-----------------+------------------+
|         author|author_post_count| author_mean_score|
+---------------+-----------------+------------------+
|       3up3down|                1|               1.0|
|      AaadamPgh|              210|14.933333333333334|
|   Aerodactyl_x|                9|11.333333333333334|
|   Aerodactyl_x|                9|11.333333333333334|
|ArmoredTricycle|               91|222.93406593406593|
+---------------+-----------------+------------------+
only showing top 5 rows



### Persist Feature-Engineered DataFrame

All non-NLP features are now attached. We persist `df_features` to disk before proceeding to NLP feature engineering, which requires UDFs and external libraries. This checkpoint ensures we don't need to re-run the aggregation-heavy feature engineering steps if the NLP work requires iteration.

In [19]:
# ── PERSIST df_features ───────────────────────────────────────────────────────

FEATURES_DIR = "../data/processed/features/"

df_features.write \
    .mode("overwrite") \
    .parquet(FEATURES_DIR)

print("Write complete. Reloading from disk...")

df_features = spark.read.parquet(FEATURES_DIR)
print(f"Rows in persisted df_features: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")
print(df_features.columns)

Write complete. Reloading from disk...
Rows in persisted df_features: 535,480,818
Columns: 27
['author', 'subreddit', 'id', 'num_comments', 'score', 'selftext', 'subreddit_id', 'title', 'created_utc', 'is_known_bot', 'is_anonymous_author', 'has_title', 'label_4class', 'label_binary', 'title_len', 'title_word_count', 'has_question', 'has_exclamation', 'is_text_post', 'selftext_len', 'hour_of_day', 'day_of_week', 'subreddit_post_count', 'subreddit_median_score', 'subreddit_median_comments', 'author_post_count', 'author_mean_score']


In [4]:
# Reload df_features

FEATURES_DIR = "../data/processed/features/"
df_features = spark.read.parquet(FEATURES_DIR)
print(f"Rows: {df_features.count():,}")
print(f"Columns: {len(df_features.columns)}")

Rows: 535,480,818
Columns: 27


In [23]:
# Check NLP library availability
try:
    from textblob import TextBlob
    print("TextBlob: available")
except ImportError:
    print("TextBlob: NOT available")

try:
    import nltk
    print("NLTK: available")
except ImportError:
    print("NLTK: NOT available")

try:
    import textstat
    print("textstat: available")
except ImportError:
    print("textstat: NOT available")

try:
    import spacy
    print("spaCy: available")
except ImportError:
    print("spaCy: NOT available")

try:
    import vaderSentiment
    print("vaderSentiment: available")
except ImportError:
    print("vaderSentiment: NOT available")

TextBlob: NOT available
NLTK: NOT available
textstat: NOT available
spaCy: NOT available
vaderSentiment: NOT available


## NLP Feature Exploration

A key element of the original project design was incorporating title-level NLP features, specifically sentiment polarity and readability to capture linguistic signals that pure structural features miss. Reddit titles vary significantly in tone: a question like "Why does nobody care about this?" carries different engagement potential than "This is the most amazing thing I've ever seen!!!"

We evaluated five NLP libraries for this task (TextBlob, NLTK, textstat, spaCy,vaderSentiment) and found none pre-installed in the Expanse Singularity container. After comparing options, we selected **VADER (Valence Aware Dictionary and sEntiment Reasoner)** as the best fit for this dataset for the following reasons:

- VADER was specifically designed for social media text and handles Reddit-specific patterns well: slang, capitalization for emphasis, repeated punctuation, and emoticons all factor into its scoring
- It returns a compound sentiment score in [-1, 1] with no model downloads or corpus dependencies required
- It is the standard choice in academic NLP work on Reddit and Twitter data
- It is lightweight enough to run as a Spark UDF without requiring GPU resources

We drop readability features (originally planned via textstat) as readability metrics were designed for longer documents and produce unreliable scores on short text like Reddit titles (typically 5–15 words).

**Performance note:** Running a Python UDF on 535M rows in local mode bypasses Spark's JVM optimizations and processes rows sequentially in Python. Benchmarking suggests this could take several hours on the full dataset. Given that the primary strength of this project is its scale (535M rows, distributed pipeline), we proceed with model training on the full dataset without NLP features for Milestone 3. NLP feature integration via a stratified sample is documented as a Milestone 4 improvement.

In [1]:
# ── INSTALL VADER ─────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["pip", "install", "vaderSentiment"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/126.0 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 26.0 MB/s eta 0:00:00




In [5]:
# ── VERIFY VADER INSTALLATION ─────────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Test on a few Reddit-style titles
test_titles = [
    "This is the most amazing thing I've ever seen!!!",
    "Why does nobody care about this?",
    "I hate everything about this",
    "lol this is hilarious",
    "Breaking news: major disaster strikes"
]

for title in test_titles:
    score = analyzer.polarity_scores(title)
    print(f"{title[:50]:<50} → compound: {score['compound']:+.3f}")

This is the most amazing thing I've ever seen!!!   → compound: +0.716
Why does nobody care about this?                   → compound: +0.494
I hate everything about this                       → compound: -0.572
lol this is hilarious                              → compound: +0.670
Breaking news: major disaster strikes              → compound: -0.800


### VADER Performance Benchmark

Before deciding whether to run VADER sentiment on the full 535M row dataset, we benchmark throughput on a small sample and extrapolate to estimate total runtime. Python UDFs in local mode process rows sequentially in Python, bypassing Spark's JVM optimizations. The benchmark will determine whether full-dataset NLP feature engineering is feasible within reasonable time constraints.

In [6]:
# ── VADER THROUGHPUT BENCHMARK ────────────────────────────────────────────────
import time

# Collect a small sample of titles to the driver
sample_size = 10000
sample_titles = df_features \
    .select("title") \
    .limit(sample_size) \
    .toPandas()["title"].tolist()

analyzer = SentimentIntensityAnalyzer()

start = time.time()
scores = [analyzer.polarity_scores(t)["compound"] for t in sample_titles]
elapsed = time.time() - start

rows_per_sec = sample_size / elapsed
estimated_total_hours = (535_480_818 / rows_per_sec) / 3600

print(f"Sample size:         {sample_size:,} rows")
print(f"Elapsed:             {elapsed:.2f} seconds")
print(f"Throughput:          {rows_per_sec:,.0f} rows/sec")
print(f"Estimated full run:  {estimated_total_hours:.1f} hours for 535M rows")

Sample size:         10,000 rows
Elapsed:             0.24 seconds
Throughput:          40,989 rows/sec
Estimated full run:  3.6 hours for 535M rows


## Model Preparation

With all non-NLP features engineered, we now prepare the DataFrame for MLlib model training. This involves four steps:

1. **Null imputation** — check for and impute any nulls in numeric feature columns
2. **Label encoding** — convert string labels to numeric indices using `StringIndexer`
3. **Feature assembly** — combine all numeric features into a single vector using `VectorAssembler`
4. **Feature scaling** — standardize the feature vector using `StandardScaler`

These steps follow feature engineering as they operate on the finalized feature set rather than raw data.